# Aula 07 — Imagem digital, filtros e Fourier

**Processamento de Linguagem Natural e Imagens · 2026.2 · Prof. James Moraes de Almeida**

Nesta aula vamos:

1. abrir, olhar, medir e salvar imagens com o **scikit-image** e o **OpenCV**;
2. entender que uma imagem é uma **matriz de números** — e que um filtro é uma conta
   feita sobre essa matriz;
3. suavizar, realçar, remover ruído e encontrar bordas;
4. levar uma imagem para o **domínio da frequência** com a transformada de Fourier;
5. resolver o enigma do início da aula: as listras que nenhum filtro tira.

> O código parte dos tutoriais 22 a 50 do curso *Python for Microscopists*, de
> Sreenivas Bhattiprolu (canal DigitalSreeni):
> https://github.com/bnsreenu/python_for_image_processing_APEER

## Antes de começar

O ambiente usado nos testes foi criado com:

```bash
conda create -n img_proc -c conda-forge python=3.12 scikit-image opencv matplotlib scipy tifffile imagecodecs pandas ipykernel
conda activate img_proc
pip install bm3d          # opcional, usado só na seção 10.6
```

As imagens estão na pasta `images/`, ao lado deste notebook. Tudo o que o notebook
grava vai para a pasta `saida/` — **nada é gravado de volta em `images/`**.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import io, img_as_float, img_as_ubyte

IMAGENS = Path("images")
SAIDA = Path("saida")
SAIDA.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 110


def mostra(*imagens, titulos=None, cmap="gray", altura=3.5):
    """Mostra várias imagens lado a lado, sem eixos."""
    n = len(imagens)
    fig, eixos = plt.subplots(1, n, figsize=(altura * 1.35 * n, altura))
    eixos = np.atleast_1d(eixos)
    for k, (ax, im) in enumerate(zip(eixos, imagens)):
        ax.imshow(im, cmap=None if im.ndim == 3 else cmap)
        ax.set_axis_off()
        if titulos:
            ax.set_title(titulos[k])
    plt.tight_layout()
    plt.show()

# 1 — Lendo imagens

## 1.1 — Com o scikit-image

`io.imread` devolve um **array do NumPy**. Repare no formato: altura × largura ×
canais.

In [ ]:
img = io.imread(IMAGENS / "Osteosarcoma_01.tif")
print("formato:", img.shape)
print("tipo:   ", img.dtype)
print("menor e maior valor:", img.min(), img.max())

In [ ]:
plt.imshow(img)
plt.axis("off")
plt.title("Células de osteossarcoma (fluorescência)")
plt.show()

Muitas funções do scikit-image preferem **ponto flutuante entre 0 e 1**. Há duas
formas de converter — e elas não dão o mesmo resultado:

In [ ]:
img_float = img_as_float(img)      # divide por 255: valores entre 0 e 1
img_cast = img.astype(float)       # só troca o tipo: valores entre 0 e 255

print("img_as_float:", img_float.min(), img_float.max())
print("astype(float):", img_cast.min(), img_cast.max())

## 1.2 — Com o OpenCV

O OpenCV também devolve um array do NumPy, mas com uma diferença que causa muito erro:
**ele guarda as cores na ordem BGR**, não RGB.

In [ ]:
img_cv = cv2.imread(str(IMAGENS / "Osteosarcoma_01.tif"))   # BGR!
img_cv_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)

mostra(img_cv, img_cv_rgb, titulos=["como o OpenCV lê (BGR)", "convertida para RGB"])

> Em 2025.2 esta seção abria a imagem com `cv2.imshow`, numa janela à parte. Aqui
> tudo é mostrado com o Matplotlib: a janela externa não funciona em Colab, em
> servidor remoto nem em alguns sistemas — e o `plt.imshow` funciona em todos.

# 2 — A imagem é uma matriz

Um tom de cinza por pixel, de 0 (preto) a 255 (branco). Vamos olhar um pedaço de
**seis por seis pixels** de uma micrografia de arenito — é exatamente o recorte que
aparece no vídeo do kernel, na aula.

In [ ]:
arenito = io.imread(IMAGENS / "sandstone.tif")
print(arenito.shape, arenito.dtype)

recorte = arenito[410:416, 224:230]
print(recorte)

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.6))
a.imshow(arenito, cmap="gray")
a.add_patch(plt.Rectangle((224, 410), 6, 6, edgecolor="orange", facecolor="none", lw=2))
a.set_title("arenito (o quadradinho laranja é o recorte)")
a.set_axis_off()
b.imshow(recorte, cmap="gray", vmin=0, vmax=255)
for (i, j), v in np.ndenumerate(recorte):
    b.text(j, i, v, ha="center", va="center", color="white" if v < 128 else "black")
b.set_title("o recorte, com os valores")
b.set_axis_off()
plt.show()

Escuro à esquerda, claro à direita: o recorte pega a **borda** de um grão. Guarde
esses números — eles voltam na seção 8.

# 3 — Filtro gaussiano e salvando a imagem

Vamos suavizar a imagem e salvá-la de cinco jeitos. Um deles dá errado — e o motivo
é instrutivo.

In [ ]:
from skimage import filters

suave = filters.gaussian(img, sigma=1, channel_axis=-1)   # float entre 0 e 1
print(suave.dtype, suave.min(), suave.max())

io.imsave(SAIDA / "salvo_com_skimage.tif", suave)
cv2.imwrite(str(SAIDA / "salvo_com_opencv.jpg"), suave)
cv2.imwrite(str(SAIDA / "salvo_com_opencv_8bits.jpg"), img_as_ubyte(suave))
cv2.imwrite(str(SAIDA / "salvo_com_opencv_rgb.jpg"),
            cv2.cvtColor(img_as_ubyte(suave), cv2.COLOR_RGB2BGR))
plt.imsave(SAIDA / "salvo_com_matplotlib.jpg", suave)
print(sorted(p.name for p in SAIDA.iterdir()))

In [ ]:
nomes = ["salvo_com_skimage.tif", "salvo_com_opencv.jpg", "salvo_com_opencv_8bits.jpg",
         "salvo_com_opencv_rgb.jpg", "salvo_com_matplotlib.jpg"]
lidas = [io.imread(SAIDA / n) for n in nomes]
mostra(*lidas, titulos=[n.replace("salvo_com_", "") for n in nomes], altura=2.6)

### Dessas imagens salvas, algumas têm uma diferença gritante. Qual é? Por que essa diferença existe?

*Duas pistas:* olhe o **tipo** do array `suave` e o intervalo dos valores; e lembre
em que ordem o OpenCV guarda as cores.

# 4 — Visualizando: mapas de cores

Uma imagem de um canal só não tem cor: o mapa de cores é uma **escolha de
visualização**. Os números são os mesmos nos quatro painéis.

In [ ]:
cinza = cv2.imread(str(IMAGENS / "Osteosarcoma_01.tif"), cv2.IMREAD_GRAYSCALE)

fig, eixos = plt.subplots(2, 2, figsize=(8, 6.4))
for ax, mapa in zip(eixos.flat, ["gray", "hot", "jet", "viridis"]):
    ax.imshow(cinza, cmap=mapa)
    ax.set_title(mapa)
    ax.set_axis_off()
plt.tight_layout()
plt.show()

O `jet` foi padrão por anos e é uma péssima escolha: ele cria bordas falsas onde o
dado não tem nenhuma (repare nas faixas amarelas). O `viridis` foi desenhado para que
a percepção de claro e escuro acompanhe o valor.

# 5 — Histogramas

O histograma conta quantos pixels há em cada nível de cinza. É a primeira coisa a
olhar numa imagem científica — antes de qualquer filtro.

In [ ]:
plt.hist(arenito.ravel(), bins=64, range=(0, 255), color="gray")
plt.xlabel("nível de cinza")
plt.ylabel("número de pixels")
plt.title("arenito")
plt.show()

In [ ]:
cores = ["red", "lightgreen", "blue"] #Coloquei lightgreen porque o red e green não consigo diferenciar.
for k, cor in enumerate(cores):
    plt.hist(img[:, :, k].ravel(), bins=64, range=(0, 150), color=cor, alpha=0.5, label=cor)
plt.legend()
plt.xlabel("intensidade")
plt.ylabel("número de pixels")
plt.title("osteossarcoma, um histograma por canal")
plt.show()

Visualizando a imagem e conferindo os histogramas, podemos escolher regiões de
interesse baseados nas intensidades de cada canal de cor. No osteossarcoma, o canal
azul é o corante dos núcleos (DAPI) — é por ele que a Aula 08 vai segmentar.

# 6 — Trabalhando com muitas imagens

Quase nunca se analisa uma imagem só. Três formas de percorrer uma pasta:

In [ ]:
import glob

arquivos = sorted(glob.glob(str(IMAGENS / "varias" / "*.tif")))
for a in arquivos:
    print(a)

In [ ]:
# processa cada uma e salva o resultado em OUTRA pasta.
# (Em 2025.2 esta célula gravava de volta na pasta que lia — e a cada execução a
#  pasta ganhava mais imagens, que a execução seguinte lia de novo.)
destino = SAIDA / "suavizadas"
destino.mkdir(exist_ok=True)

for a in arquivos:
    im = io.imread(a)
    suave = filters.gaussian(im, sigma=2, channel_axis=-1 if im.ndim == 3 else None)
    io.imsave(destino / Path(a).name, img_as_ubyte(suave), check_contrast=False)

print(sorted(p.name for p in destino.iterdir()))

In [ ]:
import os

print(os.listdir(IMAGENS / "varias"))          # só os nomes, sem caminho

for pasta, subpastas, nomes in os.walk(IMAGENS):   # percorre também as subpastas
    print(pasta, "→", len(nomes), "arquivos")

# 7 — Transformações

## 7.1 — Mudando o tamanho

In [ ]:
from skimage.transform import rescale, resize, downscale_local_mean

menor = rescale(arenito, 0.25, anti_aliasing=True)
fixa = resize(arenito, (200, 200))
media = downscale_local_mean(arenito, (4, 3))

for nome, m in [("original", arenito), ("rescale 0,25", menor),
                ("resize 200×200", fixa), ("média em blocos 4×3", media)]:
    print(f"{nome:22s} {m.shape}")
mostra(arenito, menor, fixa, media,
       titulos=["original", "rescale", "resize", "média em blocos"], altura=2.6)

## 7.2 — Separando e juntando os canais de cor

In [ ]:
mona = cv2.cvtColor(cv2.imread(str(IMAGENS / "monalisa.jpg")), cv2.COLOR_BGR2RGB)
r, g, b = cv2.split(mona)

mostra(mona, r, g, b, titulos=["RGB", "vermelho", "verde", "azul"], altura=3)

In [ ]:
trocada = cv2.merge([b, g, r])     # junta na ordem errada de propósito
mostra(mona, trocada, titulos=["original", "canais na ordem B, G, R"])

Você pode quebrar a imagem, tratar apenas um canal e recombinar — como acabou de ver.

# 8 — Um filtro é uma janela que desliza

O vídeo da aula fez a conta à mão sobre o recorte da seção 2. Vamos fazer a mesma,
em Python, e conferir que dá os mesmos números.

O **kernel gaussiano** 3×3 dá mais peso ao centro:

In [ ]:
kernel_gauss = np.array([[1, 2, 1],
                         [2, 4, 2],
                         [1, 2, 1]]) / 16

def desliza(bloco, kernel):
    """Multiplica a janela pelo kernel e soma, em cada posição (sem virar o kernel)."""
    kh, kw = kernel.shape
    saida = np.zeros((bloco.shape[0] - kh + 1, bloco.shape[1] - kw + 1))
    for i in range(saida.shape[0]):
        for j in range(saida.shape[1]):
            saida[i, j] = (bloco[i:i + kh, j:j + kw] * kernel).sum()
    return saida

print(np.round(desliza(recorte.astype(float), kernel_gauss)).astype(int))

Confira com o vídeo: o primeiro número é **46**. A borda continua lá, mas mais macia.

Agora o **kernel de Sobel**, que dá peso negativo à esquerda e positivo à direita —
ele mede uma *diferença*:

In [ ]:
kernel_sobel = np.array([[-1, 0, 1],
                         [-2, 0, 2],
                         [-1, 0, 1]])

print(desliza(recorte.astype(float), kernel_sobel).astype(int))

Números grandes exatamente onde o escuro vira claro. **O Sobel acha bordas.**

> **Convolução ou correlação?** A convolução, na definição matemática, *vira* o
> kernel antes de deslizar. O que fizemos (sem virar) chama-se correlação. Para um
> kernel simétrico, como o gaussiano, dá no mesmo; para o Sobel, troca o sinal. As
> bibliotecas e as redes neurais convolucionais fazem a correlação e a chamam de
> convolução — vamos encontrar isso de novo na Aula 10.

In [ ]:
from scipy import ndimage

print("correlação (sem virar):\n", ndimage.correlate(recorte.astype(float), kernel_sobel)[1:-1, 1:-1].astype(int))
print("convolução (virando):\n", ndimage.convolve(recorte.astype(float), kernel_sobel)[1:-1, 1:-1].astype(int))

## 8.1 — Na imagem inteira

In [ ]:
from skimage.filters import gaussian, sobel

g_sk = gaussian(arenito, sigma=2)
g_cv = cv2.GaussianBlur(arenito, (0, 0), sigmaX=2)
bordas = sobel(arenito)

mostra(arenito, g_sk, g_cv, bordas,
       titulos=["original", "gaussiano (skimage)", "gaussiano (OpenCV)", "Sobel"], altura=2.8)

Um **limiar** também é uma função sobre a matriz — a mais simples de todas: cada pixel
vira branco ou preto, conforme passe ou não de um valor.

In [ ]:
limiar = 100
binaria = arenito > limiar
mostra(arenito, binaria, titulos=["original", f"pixels acima de {limiar}"])

# 9 — Realce: o filtro *unsharp*

O nome engana: o *unsharp mask* **realça**. A receita é

$$\text{realçada} = \text{original} + a\,(\text{original} - \text{suavizada})$$

O termo entre parênteses é o **detalhe** — o que o gaussiano tirou.

In [ ]:
from skimage.filters import unsharp_mask

borrada = img_as_float(io.imread(IMAGENS / "sandstone_blur_2sigma.tif"))
suavizada = gaussian(borrada, sigma=2)
detalhe = borrada - suavizada
realcada = np.clip(borrada + 2 * detalhe, 0, 1)

mostra(borrada, suavizada, detalhe, realcada,
       titulos=["original (borrada)", "suavizada", "detalhe", "realçada, a = 2"], altura=2.8)

In [ ]:
pronta = unsharp_mask(borrada, radius=3, amount=2)
mostra(borrada, realcada, pronta, titulos=["original", "à mão", "unsharp_mask"])

# 10 — Remoção de ruído (*denoising*)

Temos a imagem limpa e uma cópia **da mesma cena** com ruído gaussiano. Então dá para
**medir** quanto cada filtro recupera, em vez de comparar no olho.

A medida é o **PSNR** (relação sinal-ruído de pico), em decibéis: quanto maior, mais
perto da imagem limpa. Cada 6 dB a mais é metade do erro.

Para os filtros lentos rodarem em segundos, usamos um recorte de 300×300 pixels.

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr

def canal_azul(nome):
    return img_as_float(io.imread(IMAGENS / nome)[:, :, 2])   # os núcleos

limpa = canal_azul("Osteosarcoma_01.tif")[400:700, 500:800]
ruidosa = canal_azul("Osteosarcoma_01_25Sigma_noise.tif")[400:700, 500:800]

resultados = {"ruidosa": ruidosa}      # cada filtro guarda a sua saída aqui
print(f"ruidosa: {psnr(limpa, ruidosa, data_range=1):.1f} dB")
mostra(limpa, ruidosa, titulos=["limpa", "com ruído"])

## 10.1 — Gaussiano

Média ponderada da vizinhança. Tira ruído — e borda junto.

In [ ]:
r = gaussian(ruidosa, sigma=1.5)
resultados["gaussiano"] = r
print(f"gaussiano: {psnr(limpa, r, data_range=1):.1f} dB")

## 10.2 — Mediana

Troca cada pixel pela **mediana** da vizinhança. Brilha contra ruído *sal e pimenta*
(pixels isolados totalmente pretos ou brancos), em que a média falha.

In [ ]:
sal = io.imread(IMAGENS / "Osteosarcoma_01_8bit_salt_pepper.tif")[400:700, 500:800]
mostra(sal, cv2.blur(sal, (3, 3)), cv2.medianBlur(sal, 3),
       titulos=["sal e pimenta", "média 3×3", "mediana 3×3"])

r = img_as_float(cv2.medianBlur(img_as_ubyte(ruidosa), 5))
resultados["mediana"] = r
print(f"mediana no ruído gaussiano: {psnr(limpa, r, data_range=1):.1f} dB")

## 10.3 — Bilateral

Uma média que só junta vizinhos **parecidos**: preserva as bordas.

In [ ]:
r = img_as_float(cv2.bilateralFilter(img_as_ubyte(ruidosa), d=9, sigmaColor=40, sigmaSpace=5))
resultados["bilateral"] = r
print(f"bilateral: {psnr(limpa, r, data_range=1):.1f} dB")

## 10.4 — *Non-local means*

Em vez da vizinhança, procura **pedaços parecidos em qualquer lugar da imagem** e faz
a média deles.

In [ ]:
from skimage.restoration import denoise_nl_means, estimate_sigma

sigma_est = estimate_sigma(ruidosa)
print(f"desvio do ruído estimado: {sigma_est * 255:.1f} níveis de cinza")
r = denoise_nl_means(ruidosa, h=0.8 * sigma_est, sigma=sigma_est,
                     patch_size=5, patch_distance=6, fast_mode=True)
resultados["non-local means"] = r
print(f"non-local means: {psnr(limpa, r, data_range=1):.1f} dB")

## 10.5 — Variação total

Procura a imagem mais **lisa** que ainda se pareça com a ruidosa. Tende a deixar a
imagem com aspecto de "cartoon": regiões lisas e bordas vivas.

In [ ]:
from skimage.restoration import denoise_tv_chambolle

r = denoise_tv_chambolle(ruidosa, weight=0.1)
resultados["variação total"] = r
print(f"variação total: {psnr(limpa, r, data_range=1):.1f} dB")

## 10.6 — BM3D (opcional)

Junta pedaços parecidos em pilhas 3D e filtra as pilhas. É, há anos, uma das
referências de qualidade. Precisa do pacote `bm3d` (`pip install bm3d`).

In [ ]:
try:
    import bm3d
    r = bm3d.bm3d(ruidosa, sigma_psd=sigma_est, stage_arg=bm3d.BM3DStages.ALL_STAGES)
    resultados["BM3D"] = r
    print(f"BM3D: {psnr(limpa, r, data_range=1):.1f} dB")
except ImportError:
    print("bm3d não instalado — pulando")

In [ ]:
import pandas as pd

tabela = pd.DataFrame({"PSNR (dB)": {n: psnr(limpa, r, data_range=1) for n, r in resultados.items()}})
print(tabela.sort_values("PSNR (dB)").round(1))

A mediana ganhou do BM3D? Antes de acreditar no placar, olhe **onde** ele é medido.

In [ ]:
fundo = limpa < 0.1
print(f"fração do recorte que é fundo escuro: {fundo.mean():.0%}")
print(f"viés do ruído no fundo: {255 * (ruidosa - limpa)[fundo].mean():+.1f} níveis de cinza")
print(f"viés do ruído nas células: {255 * (ruidosa - limpa)[limpa > 0.4].mean():+.1f} níveis de cinza")

Quase todo o recorte é fundo preto, e ali o ruído foi **cortado em zero** — não existe
nível de cinza negativo. A metade negativa do ruído sumiu, a positiva ficou: o fundo
ganhou um viés, e o ruído deixou de ser gaussiano. A mediana lida melhor com isso, e
como o fundo domina o recorte, domina o placar.

Vamos medir de novo, **só onde há célula**:

In [ ]:
celula = limpa > 0.3
tabela["PSNR só nas células (dB)"] = {n: psnr(limpa[celula], r[celula], data_range=1)
                                       for n, r in resultados.items()}
print(tabela.sort_values("PSNR só nas células (dB)").round(1))
ax = tabela.sort_values("PSNR só nas células (dB)").plot.barh(figsize=(7, 3.5))
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.02), ncol=2, frameon=False)
plt.xlabel("PSNR (dB) — maior é melhor")
plt.show()

E o BM3D, que é referência de qualidade, em último nas células? Ele e o non-local means
recebem o **desvio do ruído** como parâmetro — e nós passamos o estimado no recorte
inteiro, onde o fundo cortado em zero esconde metade do ruído. Medindo nas células:

In [ ]:
sigma_celulas = (ruidosa - limpa)[celula].std()
print(f"desvio do ruído: estimado no recorte {sigma_est * 255:.1f} · medido nas células {sigma_celulas * 255:.1f}")
try:
    import bm3d
    r = bm3d.bm3d(ruidosa, sigma_psd=sigma_celulas)
    resultados["BM3D, σ das células"] = r
    print(f"BM3D com o σ certo, só nas células: {psnr(limpa[celula], r[celula], data_range=1):.1f} dB")
except ImportError:
    print("bm3d não instalado — pulando")

**A ordem muda duas vezes.** O "melhor filtro" depende do que você mede — e os filtros
sofisticados dependem de você dizer a eles quanto ruído há.

Para contar núcleos, o
que importa é o que acontece nas células; para uma figura bonita, talvez o fundo.

E o PSNR mede o erro médio — ele não sabe o que é uma borda. Olhe as imagens:

In [ ]:
nomes = list(resultados)
mostra(*[resultados[n] for n in nomes], titulos=nomes, altura=2.2)

# 11 — Detecção de bordas

Todos os filtros abaixo aproximam a **derivada** da imagem — a borda é onde a
intensidade muda depressa.

In [ ]:
from skimage.filters import roberts, sobel, scharr, prewitt, farid

nomes = ["roberts", "sobel", "scharr", "prewitt", "farid"]
saidas = [f(arenito) for f in (roberts, sobel, scharr, prewitt, farid)]
mostra(*saidas, titulos=nomes, altura=2.4)

## 11.1 — Canny

O Canny vai além da derivada: suaviza, deriva, afina as bordas para um pixel de
largura e só mantém as que formam linhas contínuas.

In [ ]:
from skimage.feature import canny

c_sk = canny(arenito, sigma=2)
c_cv = cv2.Canny(arenito, 50, 150)
mostra(arenito, c_sk, c_cv, titulos=["original", "Canny (skimage)", "Canny (OpenCV)"])

# 12 — A transformada de Fourier

## 12.1 — Uma onda quadrada como soma de senoides

É a conta do vídeo da aula:

$$q(t) = \frac{4}{\pi}\left(\operatorname{sen} t + \frac{\operatorname{sen} 3t}{3} + \frac{\operatorname{sen} 5t}{5} + \cdots\right)$$

In [ ]:
t = np.linspace(0, 2 * np.pi, 2000)
quadrada = np.sign(np.sin(t))

for n_max in (1, 3, 9, 25):
    soma = sum(4 / (np.pi * n) * np.sin(n * t) for n in range(1, n_max + 1, 2))
    plt.plot(t, soma, label=f"até n = {n_max}")
plt.plot(t, quadrada, "k--", lw=1, label="onda quadrada")
plt.legend()
plt.xlabel("t")
plt.show()

print(f"pico com termos até 25: {soma.max():.3f}  ← o fenômeno de Gibbs")

## 12.2 — Em duas dimensões

Uma imagem senoidal tem, no espectro, **dois pontos** — um para cada sentido da onda.
O centro do espectro é a frequência zero (o brilho médio); quanto mais longe do
centro, mais fina a ondulação.

In [ ]:
y, x = np.mgrid[0:256, 0:256]
onda = np.sin(2 * np.pi * (8 * x + 3 * y) / 256)

espectro = np.fft.fftshift(np.fft.fft2(onda))   # fftshift: frequência zero no centro
mostra(onda, np.log1p(np.abs(espectro)), titulos=["uma senoide 2D", "o espectro (log)"])

In [ ]:
F = np.fft.fftshift(np.fft.fft2(arenito))
mostra(arenito, np.log1p(np.abs(F)), titulos=["arenito", "espectro do arenito (log)"])

## 12.3 — Filtrando no domínio da frequência

Zerar o **centro** tira as frequências baixas (passa-alta: sobram as bordas). Zerar a
**periferia** tira as altas (passa-baixa: a imagem borra). É o mesmo trabalho que o
Sobel e o gaussiano fazem — dito no outro domínio.

In [ ]:
H, W = arenito.shape
yy, xx = np.ogrid[:H, :W]
raio = np.hypot(yy - H / 2, xx - W / 2)

def filtra(mascara):
    return np.real(np.fft.ifft2(np.fft.ifftshift(F * mascara)))

passa_baixa = filtra(raio < 30)
passa_alta = filtra(raio > 30)
mostra(arenito, passa_baixa, passa_alta,
       titulos=["original", "passa-baixa (raio < 30)", "passa-alta (raio > 30)"])

# 13 — Contraste e cor

## 13.1 — Equalização e CLAHE

A **equalização** espalha os níveis de cinza para usar a faixa inteira. O **CLAHE** faz
isso por pedaços da imagem, com um limite — e não estoura o ruído.

In [ ]:
from skimage import exposure

baixo = io.imread(IMAGENS / "sandstone_low_contrast.tif")
eq = exposure.equalize_hist(baixo)
clahe = exposure.equalize_adapthist(baixo, clip_limit=0.02)

mostra(baixo, eq, clahe, titulos=["baixo contraste", "equalizada", "CLAHE"])
fig, eixos = plt.subplots(1, 3, figsize=(11, 2.6))
for ax, im, nome in zip(eixos, (img_as_float(baixo), eq, clahe), ("baixo contraste", "equalizada", "CLAHE")):
    ax.hist(im.ravel(), bins=64, range=(0, 1), color="gray")
    ax.set_title(nome)
plt.tight_layout()
plt.show()

## 13.2 — Iluminação desigual

Em pó de titânio, o centro da imagem está mais claro que os cantos. Equalizar a imagem
inteira não resolve; o CLAHE, que trabalha por regiões, resolve boa parte.

In [ ]:
ti = io.imread(IMAGENS / "Ti_powder.tif")
ti_cinza = cv2.cvtColor(ti, cv2.COLOR_RGB2GRAY) if ti.ndim == 3 else ti
mostra(ti_cinza, exposure.equalize_adapthist(ti_cinza, clip_limit=0.02),
       titulos=["pó de titânio", "CLAHE"])

## 13.3 — Espaços de cor

RGB mistura cor e brilho nos três canais. Em **HSV** (matiz, saturação, valor) o brilho
fica sozinho no canal V — e dá para equalizá-lo sem mexer nas cores.

In [ ]:
from skimage import color

hsv = color.rgb2hsv(mona)
hsv_eq = hsv.copy()
hsv_eq[:, :, 2] = exposure.equalize_hist(hsv[:, :, 2])
rgb_eq_errado = np.dstack([exposure.equalize_hist(mona[:, :, k]) for k in range(3)])

mostra(mona, color.hsv2rgb(hsv_eq), rgb_eq_errado,
       titulos=["original", "equalizando só o V", "equalizando R, G e B separados"])

# 14 — O enigma: as listras que nenhum filtro tira

No início da aula vimos uma micrografia com **listras periódicas** — interferência
elétrica, um defeito comum em microscopia eletrônica e de força atômica. Vamos
fabricar o mesmo defeito, para saber a resposta certa e poder medir.

In [ ]:
limpa = arenito.astype(float)
H, W = limpa.shape
y, x = np.mgrid[0:H, 0:W]

A, ciclos_x, ciclos_y = 40.0, 24, 30
listras = A * np.sin(2 * np.pi * (ciclos_x * x / W + ciclos_y * y / H))
suja = limpa + listras

def erro(im):
    """Erro RMS contra a imagem limpa, em níveis de cinza."""
    return np.sqrt(np.mean((im - limpa) ** 2))

print(f"período da listra: {1 / np.hypot(ciclos_x / W, ciclos_y / H):.1f} px")
print(f"erro da imagem suja: {erro(suja):.2f} níveis de cinza")
mostra(limpa, suja, titulos=["limpa", "com listras"])

## 14.1 — Tentativa: o gaussiano

In [ ]:
from scipy import ndimage

for s in (1, 2, 3, 4, 6):
    print(f"σ = {s} px → erro {erro(ndimage.gaussian_filter(suja, s)):6.2f}")

O erro desce de 28,3 para **20,5** no melhor caso e volta a subir. A listra tem
período de 14 pixels — o tamanho dos próprios grãos. Borrar o bastante para apagá-la
é borrar a rocha junto.

In [ ]:
mostra(suja, ndimage.gaussian_filter(suja, 3), ndimage.gaussian_filter(suja, 6),
       titulos=["suja", "gaussiano σ = 3", "gaussiano σ = 6"])

## 14.2 — No domínio da frequência

A listra é uma senoide só. No espectro, ela tem de ser **dois pontos**.

In [ ]:
F = np.fft.fft2(suja)
mag = np.log1p(np.abs(np.fft.fftshift(F)))
plt.figure(figsize=(7, 5))
plt.imshow(mag, cmap="gray")
plt.title("espectro da imagem suja (log)")
plt.axis("off")
plt.show()

In [ ]:
# procura o pico mais forte fora do centro (o centro é o brilho médio)
mag_bruta = np.abs(F).copy()
mag_bruta[0, 0] = 0
ky, kx = np.unravel_index(np.argmax(mag_bruta), mag_bruta.shape)
ky, kx = int(ky), int(kx)
print(f"pico na linha {ky}, coluna {kx} do espectro → {ky} ciclos na altura, {kx} na largura")

mascara = np.ones_like(F)
mascara[ky, kx] = 0
mascara[-ky % H, -kx % W] = 0          # o ponto simétrico
limpa_fft = np.real(np.fft.ifft2(F * mascara))

print(f"coeficientes zerados: {int((mascara == 0).sum())} de {H * W}")
print(f"erro depois do filtro: {erro(limpa_fft):.3f} níveis de cinza")
mostra(suja, limpa_fft, limpa, titulos=["suja", "2 coeficientes zerados", "limpa"])

De 28,3 para **0,41** — setenta vezes menos erro, mexendo em dois números de
trezentos e vinte e quatro mil.

E dá para ler a **amplitude** da listra direto no espectro: para uma senoide pura,
$|F| = A\,N/2$, com $N$ o número de pixels.

In [ ]:
N = H * W
print(f"amplitude lida no espectro: {2 * np.abs(F[ky, kx]) / N:.2f}  (a injetada foi {A})")

A diferença (0,18) é o quanto a própria rocha tem naquela frequência.

### Mas na vida real…

Escolhemos um número **inteiro** de ciclos para a listra cair exatamente num
coeficiente. Uma listra de verdade não tem essa gentileza.

In [ ]:
listras_reais = A * np.sin(2 * np.pi * (24.5 * x / W + 30 * y / H))
suja_real = limpa + listras_reais
F_real = np.fft.fft2(suja_real)

m = np.abs(F_real).copy(); m[0, 0] = 0
ky, kx = np.unravel_index(np.argmax(m), m.shape)
dois_pontos = np.ones_like(F_real); dois_pontos[ky, kx] = 0; dois_pontos[-ky % H, -kx % W] = 0
print(f"zerando só 2 pontos: erro {erro(np.real(np.fft.ifft2(F_real * dois_pontos))):.2f}")

Só dois pontos já não bastam: o pico **vazou** para os vizinhos. Esse é o primeiro
exercício.

# Exercícios

1. **(o vazamento)** Na listra de 24,5 ciclos, em vez de zerar dois pontos, zere um
   pequeno **disco** em volta de cada pico. Que raio deixa o erro abaixo de 3 níveis
   de cinza? O que acontece se o raio for grande demais?
2. **(duas listras)** Some à imagem suja uma segunda listra, de outra direção e outra
   amplitude. Encontre os quatro pontos no espectro e remova as duas.
3. **(o placar do ruído)** Na seção 10, varie o σ do gaussiano de 0,5 a 4. Qual σ dá o
   maior PSNR? Faça o mesmo com o `weight` da variação total.
4. **(salvando)** Explique, com o tipo e o intervalo dos arrays, por que o arquivo
   `salvo_com_opencv.jpg` saiu como saiu.
5. **(cor)** Equalize a Monalisa no espaço **LAB** (canal L) e compare com a
   equalização no HSV.
6. **(realce)** No *unsharp mask*, que valor de `amount` começa a criar halos em volta
   dos grãos? Por que eles aparecem?

# Apêndice — Registro de imagens

Duas fotos da mesma amostra raramente saem alinhadas. A **correlação de fase** acha o
deslocamento entre elas — e faz isso no domínio da frequência.

In [ ]:
from skimage.registration import phase_cross_correlation

deslocada = np.roll(arenito, shift=(12, -7), axis=(0, 1))
desloc, erro_reg, _ = phase_cross_correlation(arenito, deslocada)
print("aplicado: (12, -7) · encontrado:", desloc)
print("a função devolve o deslocamento que DESFAZ o aplicado — por isso o sinal oposto")